In [1]:
#Importar Bibliotecas 

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window


StatementMeta(, 081dca87-bee9-4cbb-add9-4f0a2c882477, 3, Finished, Available, Finished, False)

In [7]:
#Ler LakeHouse Silver

df_silver = spark.read.table("LakeHouse_Silver.dbo.fato_ocorrencias")

# Ler dimensoes na camada gold
dim_usuario = spark.read.table("LakeHouse_Gold.dbo.dim_usuario")
dim_problema = spark.read.table("LakeHouse_Gold.dbo.dim_problemas")
dim_suporte = spark.read.table("LakeHouse_Gold.dbo.dim_suporte")

StatementMeta(, 081dca87-bee9-4cbb-add9-4f0a2c882477, 9, Finished, Available, Finished, False)

In [8]:
#Dimensao Data

dim_data = df_silver.select(
    col("data_chamado").alias("data")
).distinct()

dim_data = dim_data \
    .withColumn("ano", year("data")) \
    .withColumn("mes", month("data")) \
    .withColumn("dia", dayofmonth("data")) \
    .withColumn("trimestre", quarter("data"))

dim_data.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("LakeHouse_Gold.dbo.dim_data")

StatementMeta(, 081dca87-bee9-4cbb-add9-4f0a2c882477, 10, Finished, Available, Finished, False)

In [12]:
#Garantir Integridade Referencial

fato_gold = df_silver \
    .join(dim_usuario.select("id_usuario"),
          "id_usuario",
          "inner") \
    .join(dim_problema.select("id_problema"),
          "id_problema",
          "inner") \
    .join(dim_suporte.select("id_suporte"),
          "id_suporte",
          "inner")


StatementMeta(, 081dca87-bee9-4cbb-add9-4f0a2c882477, 14, Finished, Available, Finished, False)

In [13]:
#Adicionar Referência de Data 

fato_gold = fato_gold.join(
    dim_data,
    fato_gold.data_chamado == dim_data.data,
    "left"
)


StatementMeta(, 081dca87-bee9-4cbb-add9-4f0a2c882477, 15, Finished, Available, Finished, False)

In [14]:
#Selecionar Estrutura Final da Fato

fato_gold = fato_gold.select(
    col("nr_ocorrencia"),
    col("id_usuario"),
    col("id_problema"),
    col("id_suporte"),
    col("data_chamado"),
    col("tempo_resposta_seg"),
    col("tempo_resposta_min"),
    col("sla_atendido"),
    col("status")
)

StatementMeta(, 081dca87-bee9-4cbb-add9-4f0a2c882477, 16, Finished, Available, Finished, False)

In [16]:
#Carga inicial

#fato_gold.write \
#    .format("delta") \
#    .mode("overwrite") \
#   .saveAsTable("LakeHouse_Gold.dbo.fato_ocorrencias")

StatementMeta(, 081dca87-bee9-4cbb-add9-4f0a2c882477, 18, Finished, Available, Finished, False)

In [17]:
#Atualizacao Incremental
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(
    spark,
    "LakeHouse_Gold.dbo.fato_ocorrencias"
)

delta_table.alias("target").merge(
    fato_gold.alias("source"),
    "target.nr_ocorrencia = source.nr_ocorrencia"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

StatementMeta(, 081dca87-bee9-4cbb-add9-4f0a2c882477, 19, Finished, Available, Finished, False)